# 06 · Dados da escolha do passe — Cover Shadow Value (CSV)

Fase 8a do CRISP-DM, primeira etapa da volta à modelação para a versão 4 da métrica (D36, em curso). A v3 trata todas as linhas de passe como igualmente prováveis. A versão 4 pretende modelar a escolha do passe ("se passar, para quem?"), e esta etapa limita-se a montar e a descrever a tabela em que essa escolha se vai estudar. Não há modelo nem conclusões.

Dados: SkillCorner Open Data (licença MIT), jogos 1925299 (Brisbane Roar – Perth Glory) e 1996435 (Sydney FC – Adelaide United). A sombra de cada linha vem da v3 (cone pelo tronco) e da variante "bola" (cone virado para o portador), já calculadas em `data/processed/cover_shadow/avaliacao_v3/`. O código está em `escolha.py`; a tabela, em `data/processed/cover_shadow/escolhas_v3.parquet`, tem uma linha por passe e por opção de passe.

In [1]:
import sys

import pandas as pd
from IPython.display import Markdown, display

from footy.paths import DATA_PROCESSED, PROJECTS

sys.path.insert(0, str(PROJECTS / "cover_shadow"))  # para importar o código do projeto
import escolha as es

pd.options.display.float_format = "{:.3f}".format
PASTA = DATA_PROCESSED / "cover_shadow"
passes = pd.read_parquet(PASTA / "passes_escolha_v3.parquet")  # um passe por linha
opcoes = pd.read_parquet(PASTA / "escolhas_v3.parquet")  # um passe × opção por linha
NOMES = {1925299: "Brisbane Roar – Perth Glory", 1996435: "Sydney FC – Adelaide United"}
print(f"{len(passes)} passes e {len(opcoes)} opções, em {passes['jogo'].nunique()} jogos")

1790 passes e 14959 opções, em 2 jogos


## 1. Passes

Cada passe é uma posse (`player_possession`) com `end_type == "pass"`. O recetor visado (`player_targeted_id`) é a intenção do passador e existe mesmo quando o passe falha. A tabela guarda também `high_pass`, `pass_outcome`, `start_type` e o comprimento aproximado: o `pass_distance` da SkillCorner e, nos passes falhados (que não o têm), a distância do passador ao recetor visado no frame da escolha.

In [2]:
def por_jogo(serie_por_jogo):
    """Junta o total aos jogos, para ler as contagens lado a lado."""
    t = serie_por_jogo.unstack("jogo")
    t.columns = [str(c) for c in t.columns]
    t["total"] = t.sum(axis=1)
    return t

display(Markdown("**Resultado do passe**"))
display(por_jogo(passes.groupby(["pass_outcome", "jogo"]).size()))
display(Markdown("**`high_pass` por resultado**"))
display(por_jogo(passes.assign(high_pass=passes["high_pass"].astype(object).fillna("sem informação"))
                 .groupby(["pass_outcome", "high_pass", "jogo"]).size()))
display(Markdown("**Início da posse (`start_type`)**"))
display(por_jogo(passes.groupby(["start_type", "jogo"]).size()).sort_values("total", ascending=False))

**Resultado do passe**

,1925299,1996435,total
pass_outcome,,,
offside,5,4,9
successful,780,741,1521
unsuccessful,127,133,260


**`high_pass` por resultado**

1925299  1996435  total
pass_outcome high_pass                              
offside      sem informação        5        4      9
successful   False               689      663   1352
             True                 53       42     95
             sem informação       38       36     74
unsuccessful sem informação      127      133    260

**Início da posse (`start_type`)**

,1925299,1996435,total
start_type,,,
pass_reception,671.000,637.000,1308.000
pass_interception,62.000,81.000,143.000
recovery,62.000,55.000,117.000
keep_possession,41.000,32.000,73.000
throw_in_reception,36.000,23.000,59.000
free_kick_reception,12.000,16.000,28.000
goal_kick_reception,7.000,14.000,21.000
unknown,8.000,10.000,18.000
throw_in_interception,6.000,2.000,8.000


In [3]:
comparavel = passes.dropna(subset=["comprimento_evento"]).merge(
    opcoes[opcoes["escolhida"] == 1][["jogo", "posse_id", "comprimento"]], on=["jogo", "posse_id"])
dif = (comparavel["comprimento"] - comparavel["comprimento_evento"]).abs()
print(f"Passes com `pass_distance` e linha ao recetor visado no frame da escolha: {len(comparavel)}")
print(f"Diferença absoluta entre os dois comprimentos (m): mediana {dif.median():.2f}, "
      f"percentil 90 {dif.quantile(0.9):.2f}, máximo {dif.max():.2f}")

Passes com `pass_distance` e linha ao recetor visado no frame da escolha: 1340
Diferença absoluta entre os dois comprimentos (m): mediana 1.56, percentil 90 5.22, máximo 26.25


## 2. Momento da escolha

O frame da escolha é o último frame alinhado (5 por segundo) da posse, e o intervalo é o tempo entre esse frame e o frame do passe (`frame_end` da posse), com o tracking a 10 frames por segundo. Posses de 0 s num frame ímpar não têm nenhum frame alinhado.

In [4]:
com_frame = passes[passes["frame"].notna()]
distribuicao = com_frame.groupby(["jogo", "intervalo_s"]).size().unstack("jogo").fillna(0).astype(int)
distribuicao["total"] = distribuicao.sum(axis=1)
display(Markdown("**Passes por intervalo entre o frame da escolha e o passe (s)**"))
display(distribuicao)
sem_frame = passes[passes["frame"].isna()].groupby("jogo").size()
print("Passes sem frame alinhado:", ", ".join(f"{NOMES[j]} {n}" for j, n in sem_frame.items()))
print(f"Intervalo máximo observado: {com_frame['intervalo_s'].max():.1f} s")
print(f"Limite proposto: ≤ {es.LIMITE_INTERVALO_S} s. Passes com frame que ficam fora do limite: "
      f"{int((com_frame['intervalo_s'] > es.LIMITE_INTERVALO_S).sum())}")

**Passes por intervalo entre o frame da escolha e o passe (s)**

jogo,1925299,1996435,total
intervalo_s,,,
0.000,459,443,902
0.100,346,316,662


Passes sem frame alinhado: Brisbane Roar – Perth Glory 107, Sydney FC – Adelaide United 119
Intervalo máximo observado: 0.1 s
Limite proposto: ≤ 0.4 s. Passes com frame que ficam fora do limite: 0


## 3. Conjunto de escolhas

Em cada frame da escolha, as opções são as da métrica: todos os colegas do portador (guarda-redes incluído), sem os que estão em fora de jogo (D26) e sem os cruzamentos para a área (D27). As opções sem pose de algum defesa que as possa afetar ficam na tabela com `motivo == "sem_pose"` e a sombra vazia (D13).

In [5]:
g = passes[passes["tem_linhas"]]
display(Markdown("**Número de opções por passe**"))
display(g.groupby("jogo")["n_opcoes"].describe().T)
display(Markdown("**Opções por motivo** (`motivo` vazio = avaliada)"))
display(por_jogo(opcoes.assign(motivo=opcoes["motivo"].replace("", "avaliada")).groupby(["motivo", "jogo"]).size()))
display(Markdown("**Situação do recetor visado** (passes com linhas calculadas no frame da escolha)"))
display(por_jogo(g.assign(motivo_visado=g["motivo_visado"].replace("", "avaliado"))
                 .groupby(["motivo_visado", "jogo"]).size()))

**Número de opções por passe**

jogo,1925299,1996435
count,805.000,759.000
mean,9.554,9.576
std,0.759,0.720
min,4.000,6.000
25%,9.000,9.000
50%,10.000,10.000
75%,10.000,10.000
max,10.000,10.000


**Opções por motivo** (`motivo` vazio = avaliada)

,1925299,1996435,total
motivo,,,
avaliada,5437,5991,11428
sem_pose,2254,1277,3531


**Situação do recetor visado** (passes com linhas calculadas no frame da escolha)

,1925299,1996435,total
motivo_visado,,,
avaliado,674,690,1364
cruzamento,5,4,9
fora_de_jogo,4,10,14
sem_pose,122,55,177


## 4. Variáveis de cada opção

| Variável | Definição |
|---|---|
| `escolhida` | 1 se a opção é o recetor visado |
| `comprimento` | distância (m) do portador ao colega no frame da escolha |
| `avanco` | diferença de coordenada (m) na direção da baliza adversária (`sentido_de_ataque`); negativo = para trás |
| `valor_xt`, `ganho_xt` | ganho de xT da grelha de Karun Singh (D15): só a parte positiva (a da métrica) e com sinal |
| `recetor_marcado` | 1 se algum defesa está a marcá-lo pela regra da D33 (`a_marcar`) |
| `dist_adversario` | distância (m) do colega ao adversário mais próximo, guarda-redes incluído |
| `sombra_tronco`, `sombra_bola` | 1 − `p_chegar` da v3 e da variante "bola"; vazia nas opções sem pose |
| `n_sombras_tronco`, `n_sombras_bola` | número de defesas com sombra sobre a linha, em cada versão |
| `xpass_sc`, `score_sc`, `visado_sc` | `xpass_completion`, `passing_option_score` e `targeted` da SkillCorner, só para comparação futura |

Ficam de fora as variáveis da SkillCorner que já incluem informação sobre as defesas; `xpass_sc` e `score_sc` entram apenas para comparação. A SkillCorner liga-se ao passe pelo evento `passing_option` (`associated_player_possession_event_id` = posse do passe, `player_id` = colega); quando há mais de uma deteção para o mesmo colega, fica a que cobre o frame do passe ou a mais próxima.

In [6]:
escolhidas = opcoes[opcoes["escolhida"] == 1]
cobertura = pd.DataFrame({
    "opções": opcoes.groupby("jogo").size(),
    "com valor da SkillCorner": opcoes.groupby("jogo")["xpass_sc"].count(),
    "escolhidas": escolhidas.groupby("jogo").size(),
    "escolhidas com valor": escolhidas.groupby("jogo")["xpass_sc"].count(),
})
cobertura["opções sem valor"] = cobertura["opções"] - cobertura["com valor da SkillCorner"]
cobertura["escolhidas sem valor"] = cobertura["escolhidas"] - cobertura["escolhidas com valor"]
display(Markdown("**Ligação à SkillCorner (`passing_option`)**"))
display(cobertura.T)
com_sc = escolhidas[escolhidas["visado_sc"].notna()]
print(f"Escolhidas com deteção: `targeted` da SkillCorner é verdadeiro em {int(com_sc['visado_sc'].sum())} de {len(com_sc)}")
nao_escolhidas_visadas = opcoes[opcoes["visado_sc"].eq(True) & (opcoes["escolhida"] == 0)]
print(f"Opções não escolhidas que a SkillCorner marca como `targeted`: {len(nao_escolhidas_visadas)}")
display(Markdown("**Valores em falta por variável** (% das opções)"))
display((opcoes[list(es.VARIAVEIS)].isna().mean() * 100).to_frame("% em falta").T)

**Ligação à SkillCorner (`passing_option`)**

jogo,1925299,1996435
opções,7691,7268
com valor da SkillCorner,2212,2086
escolhidas,796,745
escolhidas com valor,774,713
opções sem valor,5479,5182
escolhidas sem valor,22,32


Escolhidas com deteção: `targeted` da SkillCorner é verdadeiro em 1487 de 1487
Opções não escolhidas que a SkillCorner marca como `targeted`: 0


**Valores em falta por variável** (% das opções)

,comprimento,avanco,valor_xt,ganho_xt,recetor_marcado,dist_adversario,sombra_tronco,sombra_bola,n_sombras_tronco,n_sombras_bola,xpass_sc,score_sc
% em falta,0.000,0.000,0.000,0.000,0.000,0.000,23.605,23.605,0.000,0.000,71.268,71.268


## 5. Contagens (funil)

Passes que sobrevivem a cada passo, por ordem: totais → rasteiros → com frame da escolha dentro do limite (e linhas calculadas) → com o recetor visado no conjunto → com todas as opções avaliáveis (sem opções excluídas por falta de pose).

`high_pass` só existe nos passes completos, por isso "rasteiro" tem três leituras: `conhecidos` (`high_pass` falso), `nao_ar` (tudo o que não é `high_pass` verdadeiro, os falhados incluídos) e `todos` (sem filtro).

No último passo há duas opções para as linhas sem pose: **A**, só passes com o conjunto completo avaliável, e **B**, manter todos os passes com o recetor visado no conjunto e marcar as opções sem pose (`motivo == "sem_pose"`).

In [7]:
def tabela_do_funil(criterio):
    t = pd.DataFrame({str(j): es.funil(g, criterio=criterio) for j, g in passes.groupby("jogo")})
    t["total"] = t.sum(axis=1)
    return t.rename(index={"todas as linhas avaliáveis": "A · todas as linhas avaliáveis",
                           "recetor visado no conjunto": "B · recetor visado no conjunto (sem pose marcada)"})

for criterio in ("conhecidos", "nao_ar", "todos"):
    display(Markdown(f"**Critério de rasteiro: `{criterio}`** (limite {es.LIMITE_INTERVALO_S} s)"))
    display(tabela_do_funil(criterio))

**Critério de rasteiro: `conhecidos`** (limite 0.4 s)

,1925299,1996435,total
passes,912,878,1790
rasteiros,689,663,1352
"frame da escolha no limite, com linhas",635,589,1224
B · recetor visado no conjunto (sem pose marcada),633,584,1217
A · todas as linhas avaliáveis,160,263,423


**Critério de rasteiro: `nao_ar`** (limite 0.4 s)

,1925299,1996435,total
passes,912,878,1790
rasteiros,859,836,1695
"frame da escolha no limite, com linhas",762,724,1486
B · recetor visado no conjunto (sem pose marcada),756,710,1466
A · todas as linhas avaliáveis,183,313,496


**Critério de rasteiro: `todos`** (limite 0.4 s)

,1925299,1996435,total
passes,912,878,1790
rasteiros,912,878,1790
"frame da escolha no limite, com linhas",805,759,1564
B · recetor visado no conjunto (sem pose marcada),796,745,1541
A · todas as linhas avaliáveis,188,330,518


In [8]:
no_funil = passes["visado_no_conjunto"] & (passes["intervalo_s"] <= es.LIMITE_INTERVALO_S)
opcoes_no_funil = opcoes["visado_no_conjunto"] & (opcoes["intervalo_s"] <= es.LIMITE_INTERVALO_S)
display(Markdown("**Opção B (critério `nao_ar`): opções sem pose por passe**"))
b = passes[no_funil & es.e_rasteiro(passes, "nao_ar")]
display(b.groupby("jogo")["n_sem_pose"].describe().T)
ob = opcoes[opcoes_no_funil & es.e_rasteiro(opcoes, "nao_ar")]
display(Markdown("**Opção B (critério `nao_ar`): proporção de opções sem pose, e do recetor visado sem pose**"))
display(pd.DataFrame({
    "opções": ob.groupby("jogo").size(),
    "% opções sem pose": 100 * (ob["motivo"] == "sem_pose").groupby(ob["jogo"]).mean(),
    "% escolhidas sem pose": 100 * (ob.loc[ob["escolhida"] == 1, "motivo"] == "sem_pose")
    .groupby(ob.loc[ob["escolhida"] == 1, "jogo"]).mean(),
}))

**Opção B (critério `nao_ar`): opções sem pose por passe**

jogo,1925299,1996435
count,756.000,710.000
mean,2.798,1.703
std,2.461,2.222
min,0.000,0.000
25%,1.000,0.000
50%,3.000,1.000
75%,4.000,3.000
max,10.000,10.000


**Opção B (critério `nao_ar`): proporção de opções sem pose, e do recetor visado sem pose**

,opções,% opções sem pose,% escolhidas sem pose
jogo,,,
1925299,7234,29.237,13.889
1996435,6819,17.730,7.183


## 6. Passes longos

Passes por faixa de comprimento (`comprimento_passe`) e percentagem pelo ar. A percentagem calcula-se sobre os passes com `high_pass` conhecido; a coluna `sem_informacao` conta os restantes (passes falhados e fora de jogo).

In [9]:
for nome, dados in [*[(NOMES[j], g) for j, g in passes.groupby("jogo")], ("Os dois jogos", passes)]:
    display(Markdown(f"**{nome}**"))
    display(es.faixas_de_comprimento(dados))

**Brisbane Roar – Perth Glory**

,passes,pelo_ar,sem_informacao,pct_pelo_ar
comprimento_passe,,,,
"[0.0, 10.0)",197,4,42,2.581
"[10.0, 20.0)",408,11,45,3.030
"[20.0, 30.0)",186,12,29,7.643
"[30.0, 35.0)",44,5,10,14.706
"[35.0, 40.0)",20,8,3,47.059
"[40.0, inf)",30,13,14,81.250


**Sydney FC – Adelaide United**

,passes,pelo_ar,sem_informacao,pct_pelo_ar
comprimento_passe,,,,
"[0.0, 10.0)",227,5,37,2.632
"[10.0, 20.0)",429,7,64,1.918
"[20.0, 30.0)",122,10,25,10.309
"[30.0, 35.0)",34,6,4,20.000
"[35.0, 40.0)",21,9,5,56.250
"[40.0, inf)",16,5,9,71.429


**Os dois jogos**

,passes,pelo_ar,sem_informacao,pct_pelo_ar
comprimento_passe,,,,
"[0.0, 10.0)",424,9,79,2.609
"[10.0, 20.0)",837,18,109,2.473
"[20.0, 30.0)",308,22,54,8.661
"[30.0, 35.0)",78,11,14,17.188
"[35.0, 40.0)",41,17,8,51.515
"[40.0, inf)",46,18,23,78.261


## 7. Descrição da tabela

Opções por passe, média de cada variável nas opções escolhidas e não escolhidas, e correlações entre a sombra e as restantes variáveis. Consideram-se os passes com frame da escolha dentro do limite e o recetor visado no conjunto; a primeira leitura inclui todos os passes e a segunda só os rasteiros com `high_pass` falso. Os valores em falta (sombra nas opções sem pose, SkillCorner nas opções não detetadas) ficam fora de cada média.

In [10]:
base = opcoes[opcoes_no_funil]
leituras = {"todos os passes": base, "só passes rasteiros (`high_pass` falso)": base[es.e_rasteiro(base, "conhecidos")]}
for nome, dados in leituras.items():
    por_passe = dados.groupby(["jogo", "posse_id"]).size()
    display(Markdown(f"### {nome}: {por_passe.size} passes, {len(dados)} opções"))
    display(Markdown("**Opções por passe**"))
    display(por_passe.describe().to_frame("opções por passe").T)
    display(Markdown("**Média por opção**"))
    medias = es.medias_por_escolha(dados)
    medias["n (não escolhidas)"] = [int(dados.loc[dados["escolhida"] == 0, v].count()) for v in medias.index]
    medias["n (escolhidas)"] = [int(dados.loc[dados["escolhida"] == 1, v].count()) for v in medias.index]
    display(medias)
    display(Markdown("**Correlação (Pearson) da sombra com as restantes variáveis, nas opções com sombra avaliada**"))
    display(es.correlacao_sombra_marcacao(dados))

### todos os passes: 1541 passes, 14777 opções

**Opções por passe**

,count,mean,std,min,25%,50%,75%,max
opções por passe,1541.000,9.589,0.703,6.000,9.000,10.000,10.000,10.000


**Média por opção**

escolhida,não escolhidas,escolhidas,n (não escolhidas),n (escolhidas)
comprimento,26.536,16.077,13236,1541
avanco,0.922,2.995,13236,1541
valor_xt,0.004,0.004,13236,1541
ganho_xt,0.002,0.003,13236,1541
recetor_marcado,0.167,0.101,13236,1541
dist_adversario,9.601,9.038,13236,1541
sombra_tronco,0.241,0.068,9912,1364
sombra_bola,0.260,0.084,9912,1364
n_sombras_tronco,0.654,0.232,13236,1541
n_sombras_bola,0.487,0.206,13236,1541


**Correlação (Pearson) da sombra com as restantes variáveis, nas opções com sombra avaliada**

... (4 outputs left out)

**Média por opção**

escolhida,não escolhidas,escolhidas,n (não escolhidas),n (escolhidas)
comprimento,26.925,14.968,10411,1217
avanco,1.492,0.996,10411,1217
valor_xt,0.004,0.002,10411,1217
ganho_xt,0.002,0.001,10411,1217
recetor_marcado,0.173,0.075,10411,1217
dist_adversario,9.756,9.832,10411,1217
sombra_tronco,0.239,0.037,7871,1130
sombra_bola,0.261,0.044,7871,1130
n_sombras_tronco,0.649,0.159,10411,1217
n_sombras_bola,0.494,0.131,10411,1217


**Correlação (Pearson) da sombra com as restantes variáveis, nas opções com sombra avaliada**

,sombra_tronco,sombra_bola
recetor_marcado,0.048,0.050
comprimento,0.222,0.233
dist_adversario,-0.158,-0.162
avanco,0.170,0.206


In [11]:
avaliadas = base.dropna(subset=["sombra_tronco"])
display(Markdown("**Sombra (v3) por recetor marcado e não marcado, nas opções avaliadas**"))
display(avaliadas.groupby("recetor_marcado")["sombra_tronco"].describe())
display(Markdown("**Opções avaliadas por recetor marcado e `escolhida`**"))
display(pd.crosstab(avaliadas["recetor_marcado"], avaliadas["escolhida"]))

**Sombra (v3) por recetor marcado e não marcado, nas opções avaliadas**

,count,mean,std,min,25%,50%,75%,max
recetor_marcado,,,,,,,,
0,9435.000,0.214,0.350,0.000,0.000,0.000,0.322,1.000
1,1841.000,0.250,0.365,0.000,0.000,0.004,0.494,1.000


**Opções avaliadas por recetor marcado e `escolhida`**

escolhida,0,1
recetor_marcado,,
0,8200,1235
1,1712,129
